# 00.05 — Partial Derivatives & the Chain Rule

**You should already know:** vectors and dot products (00.01), matrix notation (00.02), eigenvectors (00.03), and the derivative and gradient ideas from 00.04.  
**By the end, you will be able to:** compute first-order partial derivatives, separate inner and outer functions, apply a one-intermediate-variable chain rule, verify it by expansion, and connect it to a small loss-gradient example.  
**Difficulty:** ★★☆☆☆ · **Time:** about 35–45 minutes

All examples use dimensionless, synthetic teaching values. This lesson does not teach backpropagation, automatic differentiation, gradient descent, or model evaluation.

## 1. The Problem

Many calculations are built in stages: inputs first produce an intermediate quantity, and that intermediate quantity produces the final output. If one input changes, how can we trace its effect through every stage without expanding the whole expression?

The chain rule gives a systematic answer. We will use one small composition, verify it a second way, and then connect the pattern to the fixed loss example from 00.04.

## 2. The Toy Example

Let `x` and `y` be independent scalar inputs:

$$u=g(x,y)=x+2y.$$

Let the outer function be:

$$h(u)=u^2.$$

The composite output is:

$$f(x,y)=h(g(x,y))=(x+2y)^2.$$

We freeze the teaching point at `(x,y)=(3,2)`. The values are dimensionless scalars. Later NumPy gradients use one-dimensional arrays of shape `(2,)`.

## 3. Intuition Before Math

There are two rates to trace:

1. How fast does the intermediate `u` change when `x` or `y` changes?
2. How fast does the outer output change when `u` changes?

Along one dependency path, these rates multiply. Here, changing `y` by one unit changes `u` by two units. At the frozen point, changing `u` locally changes `f` at rate `14`, so the composite `y`-rate is `14 × 2 = 28`.

For a partial derivative, change one named input while holding the other independent input fixed. The symbols are not ordinary fractions that can always be cancelled, especially when several dependency paths contribute.

## 4. Mathematical Derivation

### Inner partial derivatives

Holding `y` fixed:

$$\frac{\partial u}{\partial x}
=\frac{\partial}{\partial x}(x+2y)=1+0=1.$$

Holding `x` fixed:

$$\frac{\partial u}{\partial y}
=\frac{\partial}{\partial y}(x+2y)=0+2=2.$$

The value `2` comes from the coefficient of `y`, not from substituting the coordinate `y=2`.

### Outer derivative and chain rule

Since `h(u)=u²`,

$$\frac{df}{du}=h'(u)=2u.$$

The single-path chain rule gives:

$$\frac{\partial f}{\partial x}
=\frac{df}{du}\frac{\partial u}{\partial x}
=2u(1)=2x+4y,$$

$$\frac{\partial f}{\partial y}
=\frac{df}{du}\frac{\partial u}{\partial y}
=2u(2)=4x+8y.$$

In vector form:

$$\nabla u=[\partial u/\partial x,\partial u/\partial y]=[1,2],\qquad
\nabla f=\frac{df}{du}\nabla u.$$

At `(3,2)`, `u=7`, so `df/du=14` and `∇f=[14,28]`.

### Independent expansion

$$f(x,y)=(x+2y)^2=x^2+4xy+4y^2.$$

Differentiating this expanded expression directly:

$$\frac{\partial f}{\partial x}=2x+4y,\qquad
\frac{\partial f}{\partial y}=4x+8y.$$

This agrees symbolically with the chain-rule result. If an output depends on two intermediate variables, contributions add:

$$\frac{\partial z}{\partial x}
=\frac{\partial z}{\partial u}\frac{\partial u}{\partial x}
+\frac{\partial z}{\partial v}\frac{\partial v}{\partial x}.$$

This is a boundary illustration, not a backpropagation or Jacobian lesson.

## 5. Manual Walkthrough

At `(x,y)=(3,2)`:

| Quantity | Arithmetic | Result |
|---|---|---:|
| Weighted term | `2 × 2` | `4` |
| Intermediate | `3 + 4` | `u = 7` |
| Output | `7 × 7` | `f = 49` |
| Outer rate | `2 × 7` | `df/du = 14` |
| Inner x-rate | coefficient of `x` | `∂u/∂x = 1` |
| Inner y-rate | coefficient of `y` | `∂u/∂y = 2` |
| Composite x-rate | `14 × 1` | `∂f/∂x = 14` |
| Composite y-rate | `14 × 2` | `∂f/∂y = 28` |
| Gradient | `[14, 28]` | `∇f = [14,28]` |

The expanded arithmetic independently gives:

```text
f = 3² + 4×3×2 + 4×2² = 9 + 24 + 16 = 49
∂f/∂x = 2×3 + 4×2 = 6 + 8 = 14
∂f/∂y = 4×3 + 8×2 = 12 + 16 = 28
```

### Bridge to 00.04

In the earlier fixed example, `r = w @ x - y_target` is the inner quantity and `L(r)=½r²` is the outer quantity:

```text
x = [1, 2], w = [1, 1], y_target = 5
prediction = 3, residual = -2, loss = 2
dL/dr = -2, ∇w r = [1, 2]
∇w L = (-2)[1, 2] = [-2, -4]
```

This is a structural connection to the known result, not an optimization update.

## 6. Teaching Implementation

The implementation mirrors the two independent mathematical paths. Scalars have shape `()` and gradients have shape `(2,)`.

In [ ]:
import numpy as np

RTOL = 1e-12
ATOL = 1e-12

x = np.float64(3.0)
y = np.float64(2.0)
u = x + 2.0 * y
f = u * u

outer_rate = 2.0 * u
inner_gradient = np.array([1.0, 2.0], dtype=np.float64)
chain_gradient = outer_rate * inner_gradient

expanded_output = x * x + 4.0 * x * y + 4.0 * y * y
expanded_gradient = np.array(
    [2.0 * x + 4.0 * y, 4.0 * x + 8.0 * y], dtype=np.float64
)

print(f"u={u}, f={f}, chain gradient={chain_gradient.tolist()}")
print(f"expanded output={expanded_output}, expanded gradient={expanded_gradient.tolist()}")

## 7. Verification of Fixed Teaching Calculations

These assertions verify the frozen forward values, derivative values, shapes, and independent expansion path. The final output is `PASS` only after every assertion succeeds.

In [ ]:
assert np.asarray(u).shape == ()
assert np.asarray(f).shape == ()
assert np.asarray(outer_rate).shape == ()
assert inner_gradient.shape == (2,)
assert chain_gradient.shape == (2,)
assert expanded_gradient.shape == (2,)
assert chain_gradient.dtype == np.float64
assert expanded_gradient.dtype == np.float64

np.testing.assert_allclose(u, 7.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(f, 49.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(outer_rate, 14.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(chain_gradient, [14.0, 28.0], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(expanded_output, 49.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(expanded_gradient, [14.0, 28.0], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(
    chain_gradient, expanded_gradient, rtol=RTOL, atol=ATOL
)

bridge_x = np.array([1.0, 2.0], dtype=np.float64)
weights = np.array([1.0, 1.0], dtype=np.float64)
y_target = np.float64(5.0)
prediction = weights @ bridge_x
residual = prediction - y_target
loss = 0.5 * residual * residual
loss_gradient = residual * bridge_x

np.testing.assert_allclose(prediction, 3.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(residual, -2.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(loss, 2.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(loss_gradient, [-2.0, -4.0], rtol=RTOL, atol=ATOL)
print("PASS")

## 8. Independent Cross-Check

The expanded-polynomial calculation above does not call the chain-rule calculation. The following deterministic points test both expressions over different quadrants:

```text
(0,0), (1,-1), (-2,3), (4.5,1.5), (-3,-2)
```

In [ ]:
points = np.array(
    [[0.0, 0.0], [1.0, -1.0], [-2.0, 3.0], [4.5, 1.5], [-3.0, -2.0]],
    dtype=np.float64,
)

for point_x, point_y in points:
    intermediate = point_x + 2.0 * point_y
    chain = (2.0 * intermediate) * np.array([1.0, 2.0])
    direct = np.array(
        [2.0 * point_x + 4.0 * point_y, 4.0 * point_x + 8.0 * point_y]
    )
    np.testing.assert_allclose(chain, direct, rtol=RTOL, atol=ATOL)

print("Independent expanded-polynomial cross-check: PASS")

## 9. Visual Explanation or Diagnostics

![Two side-by-side plots explain one chain-rule composition at (x,y)=(3,2). The left plot shows diagonal level lines of u=x+2y, with the highlighted line u=7 passing through (3,2). Two arrows start at that point and point perpendicular to the highlighted line: a blue display-scaled arrow labelled gradient u=[1,2], and an orange display-scaled arrow labelled gradient f=[14,28]; the orange arrow is longer but points in the same direction. The right plot shows the upward-opening curve f(u)=u², the point (u,f)=(7,49), and an orange dashed tangent whose slope is labelled df/du=14. Text states that the outer rate amplifies inner rates [1,2] to [14,28].](assets/partial_derivatives_chain_rule.png)

**Caption:** The left panel shows level lines of `u=x+2y` and the frozen point `(3,2)`. Display-scaled arrows for `∇u=[1,2]` and `∇f=[14,28]` are perpendicular to the highlighted line `u=7`; the right panel shows `f(u)=u²` and its tangent at `(7,49)` with slope `14`. The arrows are explanatory display scalings, not gradient-descent steps or model-quality evidence.

## 10. Failure Modes & Pitfalls

- **Dropping a cross-term:** when differentiating `4xy` with respect to `x`, hold `y` fixed but retain the coefficient `4y`, so the result is `4y`.
- **Using the wrong evaluation value:** `df/du=2u` is evaluated at `u=7`, not at `x=3`, giving `14`.
- **Omitting an inner rate:** `∂f/∂y` is `14×2=28`, not `14`.
- **Cancelling partial symbols across paths:** multiple dependency paths contribute a sum; partial symbols are not freely cancellable fractions.
- **Confusing formal and composite partials:** holding an intermediate variable fixed differs from holding an underlying input fixed while the intermediate changes.
- **Treating a gradient as an update:** this lesson gives local sensitivity only; learning rates and optimization are deferred.

### Spot the bug

A learner writes `df/du=2x=6` and `∇f(3,2)=[6,12]`. The correct intermediate value is `u=7`, so `df/du=14`; multiplying by `[∂u/∂x,∂u/∂y]=[1,2]` gives `∇f=[14,28]`.

## 11. Exercises

### 1. Direct partials
For `g(x,y)=3x²−2xy+5y²` at `(2,3)`, calculate `g` and its ordered gradient.

<details><summary>Answer</summary>

`g=45`, `∂g/∂x=6`, `∂g/∂y=26`, so `∇g=[6,26]`.
</details>

### 2. Single-path cubic
Let `u=3x−y` and `f=u³`. At `(2,4)`, calculate the chain-rule gradient and verify it by expanding `(3x−y)³`.

<details><summary>Answer</summary>

`u=2`, `df/du=12`, `[∂u/∂x,∂u/∂y]=[3,−1]`, so `∇f=[36,−12]`. The expansion gives the same values.
</details>

### 3. Two-branch rule
Let `u=x+y`, `v=2x−y`, and `z=u²+v²`. At `(1,2)`, compute the gradient by adding both paths and verify by expansion.

<details><summary>Answer</summary>

`u=3`, `v=0`, so `∇z=[6,6]`. Direct expansion gives `z=5x²−2xy+2y²`, whose partials also evaluate to `[6,6]`.
</details>

## 12. References and Further Reading

1. Strang, G. and Herman, E. J. *Calculus Volume 3*, §§4.3 and 4.5. OpenStax. [Partial derivatives](https://openstax.org/books/calculus-volume-3/pages/4-3-partial-derivatives) and [the chain rule](https://openstax.org/books/calculus-volume-3/pages/4-5-the-chain-rule).
2. Auroux, D. *18.02 Multivariable Calculus*. MIT OpenCourseWare. [Course materials](https://ocw.mit.edu/courses/18-02-multivariable-calculus-fall-2007/).
3. Deisenroth, M. P., Faisal, A. A., and Ong, C. S. *Mathematics for Machine Learning*, Chapter 5. [Official site](https://mml-book.com/).
4. NumPy Developers. [NumPy 1.26 reference documentation](https://numpy.org/doc/1.26/).